# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     |Fernando Gonzalez   |
| **Fecha**      19-sep-26|   |
| **Expediente**748126 |   |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

¿Por qué?

- **para separar los datos en entrenamiento y prueba. Así podemos entrenar el modelo con una parte de los datos y evaluar qué tan bien funciona con datos que no utilizó durante el entrenamiento.**

Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?

- **utilizar todos los datos para entrenar no nos permitiría tener datos independientes para evaluar el modelo. Por eso se separan los datos, para poder medir qué tan bien generaliza el modelo.**

El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### Ejercicio 1

Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.

In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score

data = pd.read_excel("Motor+Trend+Car+Road+Tests.xlsx")

data = data.drop(columns=["model"])

X = data.drop(columns=["mpg"])
y = data["mpg"]

modelo = LinearRegression()

loo = LeaveOneOut()

mse = -cross_val_score(modelo, X, y, cv=loo,
                       scoring="neg_mean_squared_error")

print("MSE promedio:", mse.mean())
print("Desviación estándar:", mse.std())

MSE promedio: 12.181558006901959
Desviación estándar: 17.06739987188857


Interpreta.

- **El MSE promedio fue de 12.18. Esto representa el error cuadrático medio de las predicciones usando LOOCV. La desviación estándar fue de 17.07, lo que indica que el error cambia entre las diferentes observaciones. En general, el modelo tiene un error promedio de 12.18 al predecir mpg.**

## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### Ejercicio 2
Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

modelo = LinearRegression()

kf = KFold(n_splits=10, shuffle=True, random_state=1)

mse = -cross_val_score(modelo, X, y, cv=kf,
                       scoring="neg_mean_squared_error")

print("MSE promedio:", mse.mean())
print("Desviación estándar:", mse.std())

MSE promedio: 11.633486646766011
Desviación estándar: 5.993662365220403


- **El dataset se dividió en 10 partes para realizar la validación cruzada. En cada iteración se utilizaron 9 partes para entrenar y 1 para probar. El MSE permite medir el error de las predicciones.**

Interpreta.

- **El MSE promedio representa el error promedio del modelo al realizar la validación cruzada. La desviación estándar muestra cuánto cambia el error entre los diferentes grupos.**

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3